# Proyecto Final — DS_Telecom (Interconnect)
**Objetivo:** construir un modelo de clasificación que prediga si un cliente cancelará el servicio (churn).  
Si se identifica a tiempo que un cliente planea irse, el equipo de marketing podrá ofrecer promociones o planes especiales para retenerlo.

## Datos disponibles
Los datos provienen de 4 fuentes y se relacionan mediante la columna **`customerID`**:
- `contract.csv`: información del contrato (fechas, tipo de contrato, pagos, cargos).
- `personal.csv`: datos personales del cliente.
- `internet.csv`: servicios relacionados con internet.
- `phone.csv`: servicios telefónicos.

**Ruta de los datos en la plataforma:** `/datasets/final_provider/`

## Variable objetivo
Definiremos la variable objetivo a partir de la columna **`EndDate`** del archivo de contrato:
- `EndDate = 'No'` → el cliente **no** ha cancelado.
- `EndDate ≠ 'No'` → el cliente **sí** ha cancelado.

## Métricas de evaluación
- **Métrica principal:** AUC-ROC  
- **Métrica adicional:** exactitud (accuracy)

## Plan general de trabajo (alto nivel)
1. **Carga y exploración inicial:** revisar estructura, tipos de datos y valores ausentes.
2. **Preparación de datos:** unir tablas por `customerID`, limpiar y transformar variables (fechas y categóricas).
3. **Modelado y evaluación:** entrenar modelos, comparar métricas (AUC-ROC y accuracy) y ajustar el mejor.
4. **Conclusiones:** resumir hallazgos, limitaciones y recomendaciones para negocio.


In [1]:
import pandas as pd
import numpy as np


In [2]:
path = '/datasets/final_provider/'

contract = pd.read_csv(path + 'contract.csv')
personal = pd.read_csv(path + 'personal.csv')
internet = pd.read_csv(path + 'internet.csv')
phone = pd.read_csv(path + 'phone.csv')


In [3]:
contract.head(), contract.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB


(   customerID   BeginDate              EndDate            Type  \
 0  7590-VHVEG  2020-01-01                   No  Month-to-month   
 1  5575-GNVDE  2017-04-01                   No        One year   
 2  3668-QPYBK  2019-10-01  2019-12-01 00:00:00  Month-to-month   
 3  7795-CFOCW  2016-05-01                   No        One year   
 4  9237-HQITU  2019-09-01  2019-11-01 00:00:00  Month-to-month   
 
   PaperlessBilling              PaymentMethod  MonthlyCharges TotalCharges  
 0              Yes           Electronic check           29.85        29.85  
 1               No               Mailed check           56.95       1889.5  
 2              Yes               Mailed check           53.85       108.15  
 3               No  Bank transfer (automatic)           42.30      1840.75  
 4              Yes           Electronic check           70.70       151.65  ,
 None)

In [4]:
personal.head(), personal.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB


(   customerID  gender  SeniorCitizen Partner Dependents
 0  7590-VHVEG  Female              0     Yes         No
 1  5575-GNVDE    Male              0      No         No
 2  3668-QPYBK    Male              0      No         No
 3  7795-CFOCW    Male              0      No         No
 4  9237-HQITU  Female              0      No         No,
 None)

In [5]:
internet.head(), internet.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB


(   customerID InternetService OnlineSecurity OnlineBackup DeviceProtection  \
 0  7590-VHVEG             DSL             No          Yes               No   
 1  5575-GNVDE             DSL            Yes           No              Yes   
 2  3668-QPYBK             DSL            Yes          Yes               No   
 3  7795-CFOCW             DSL            Yes           No              Yes   
 4  9237-HQITU     Fiber optic             No           No               No   
 
   TechSupport StreamingTV StreamingMovies  
 0          No          No              No  
 1          No          No              No  
 2          No          No              No  
 3         Yes          No              No  
 4          No          No              No  ,
 None)

In [6]:
phone.head(), phone.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB


(   customerID MultipleLines
 0  5575-GNVDE            No
 1  3668-QPYBK            No
 2  9237-HQITU            No
 3  9305-CDSKC           Yes
 4  1452-KIOVK           Yes,
 None)

## Revisión inicial de los datos (carga y estructura)

Se cargaron correctamente los 4 archivos desde `/datasets/final_provider/`:

- **contract**: 7043 filas y 8 columnas. Contiene fechas (`BeginDate`, `EndDate`), tipo de contrato, facturación sin papel, método de pago y cargos (`MonthlyCharges`, `TotalCharges`).
- **personal**: 7043 filas y 5 columnas. Contiene variables demográficas (género, `SeniorCitizen`, pareja y dependientes).
- **internet**: 5517 filas y 8 columnas. Contiene información de servicios de internet (seguridad, backup, soporte técnico, streaming).
- **phone**: 6361 filas y 2 columnas. Contiene información de servicio telefónico (`MultipleLines`).

### Observaciones sobre tipos de datos
- Variables numéricas ya correctamente tipadas:
  - `MonthlyCharges` es `float64`.
  - `SeniorCitizen` es `int64`.
- Variables categóricas se encuentran como `object` (por ejemplo, `Yes/No`, tipo de servicio, método de pago), lo cual es esperable.
- Variables que requerirán limpieza/conversión en la etapa de preparación de datos:
  - `BeginDate` y `EndDate` están como `object` y deberán convertirse a `datetime` (considerando que `EndDate` contiene el valor `'No'` para clientes activos).
  - `TotalCharges` está como `object` y deberá convertirse a numérica antes del modelado.

En esta etapa solo se verificó la carga y estructura; la limpieza y transformación se realizarán posteriormente.


In [7]:
contract['churn'] = contract['EndDate'].apply(lambda x: 0 if x == 'No' else 1)


In [8]:
contract['churn'].value_counts()


0    5174
1    1869
Name: churn, dtype: int64

In [9]:
data = contract.merge(personal, on='customerID', how='left')
data = data.merge(internet, on='customerID', how='left')
data = data.merge(phone, on='customerID', how='left')


In [10]:
data.shape


(7043, 21)

## Integración de las fuentes de datos

Las distintas fuentes de información fueron unificadas en un solo DataFrame utilizando la columna `customerID` como clave primaria.

- Se tomó como base la tabla **contract**, ya que contiene a todos los clientes activos e inactivos.
- Las tablas **personal**, **internet** y **phone** se unieron mediante un `left join` para evitar la pérdida de clientes.
- Como resultado, el DataFrame final contiene **7043 filas (clientes)** y **21 columnas**.

Los valores faltantes que aparecen tras la unión corresponden a clientes que no cuentan con ciertos servicios (por ejemplo, internet o telefonía), lo cual es consistente con el contexto del negocio.


In [11]:
data.isna().sum().sort_values(ascending=False)


StreamingMovies     1526
StreamingTV         1526
TechSupport         1526
DeviceProtection    1526
OnlineBackup        1526
OnlineSecurity      1526
InternetService     1526
MultipleLines        682
Partner                0
Dependents             0
customerID             0
BeginDate              0
gender                 0
churn                  0
TotalCharges           0
MonthlyCharges         0
PaymentMethod          0
PaperlessBilling       0
Type                   0
EndDate                0
SeniorCitizen          0
dtype: int64

## Revisión de valores faltantes

Tras la integración de las fuentes de datos, se revisaron los valores faltantes en el DataFrame final.

- Se identificaron **1526 valores faltantes** en las variables relacionadas con servicios de internet (`InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`), correspondientes a clientes que no cuentan con este tipo de servicio.
- Se encontraron **682 valores faltantes** en la variable `MultipleLines`, asociados a clientes sin servicio telefónico.
- No se detectaron valores faltantes en la variable objetivo (`churn`), en los cargos (`MonthlyCharges`, `TotalCharges`) ni en los datos personales o contractuales.

Estos valores faltantes son coherentes con el contexto del negocio y serán tratados adecuadamente en la etapa de preparación de datos.


## Preguntas aclaratorias para el líder del equipo

1. ¿Se debe considerar únicamente la columna `EndDate` para definir la cancelación del servicio o existe alguna otra condición de negocio que indique churn?
2. En el caso de clientes sin servicios de internet o telefonía, ¿se espera que los valores faltantes se traten como una categoría explícita (por ejemplo, "No service") o simplemente como ausencia de servicio?
3. ¿Existe algún periodo mínimo de antigüedad del cliente que deba considerarse para el análisis (por ejemplo, excluir clientes muy recientes)?
4. ¿Se prioriza la interpretabilidad del modelo sobre el rendimiento (AUC-ROC), o el objetivo principal es maximizar la métrica aun si el modelo es más complejo?
5. ¿Hay restricciones o preferencias respecto a los modelos a utilizar (por ejemplo, modelos lineales vs. modelos basados en árboles)?


## Plan aproximado para resolver la tarea

1. **Preparación y limpieza de datos**  
   Unificar todas las fuentes en un solo DataFrame, corregir tipos de datos (fechas y variables numéricas) y tratar los valores faltantes de acuerdo con el contexto del negocio.

2. **Análisis exploratorio y selección de variables**  
   Analizar la relación entre las variables y la cancelación del servicio, identificando patrones relevantes y posibles variables predictoras.

3. **Entrenamiento y evaluación de modelos**  
   Entrenar distintos modelos de clasificación, evaluarlos principalmente con la métrica AUC-ROC y seleccionar el modelo con mejor desempeño.

4. **Ajuste final y validación**  
   Ajustar hiperparámetros del modelo seleccionado y validar su rendimiento para asegurar estabilidad y capacidad de generalización.

5. **Conclusiones y recomendaciones**  
   Interpretar los resultados obtenidos, identificar limitaciones del análisis y proponer recomendaciones accionables para el negocio.


<div class="alert alert-block alert-success">
<b>Comentario general (1ra Iteracion)</b> <a class=“tocSkip”></a>

Hola, Diego. Excelente explicación sobre tu plan de trabajo para tu proyecto final! Muy bien ilustrado y explicado solamente te recomiendo el uso de diagramas.

Agregas un apartado de preparación y limpieza de datos para fortalecer nuestro análisis. Asi como analizar la distribución e las variables y creación de nuevas variables que serán de utilidad para el análisis. 

Comentas sobre un Análisis Exploratorio de Datos (EDA) completo, incluyendo gráficas representativas y conclusiones relevantes para cada una. Recuerda que es importante verificar el balance de clases en la variable objetivo. En caso de encontrar un desbalance significativo, considera ajustar los parámetros de los modelos para manejar esta situación o aplicar técnicas de balanceo como oversampling o undersampling. Solamente te recomiendo colocar la matriz de correlación para identificar posibles colinealidades. 

Si las métricas (por ejemplo, F1-score, accuracy, etc.) son altas en el conjunto de entrenamiento pero bajas en el de prueba, esto indica un posible sobreajuste (overfitting). En tal caso, será necesario ajustar el proceso de entrenamiento para reducir este problema.

Como referencia, puedes utilizar un umbral de 0.75 en el F1-score para considerar que el modelo tiene un desempeño adecuado. También se recomienda realizar una prueba de cordura utilizando un DummyClassifier como modelo base. Finalmente, asegúrate de identificar correctamente si el problema que estás abordando es de regresión o de clasificación, ya que esto determinará la elección del modelo y las métricas de evaluación adecuadas.
